In [ ]:
%pip install python-dotenv kaggle ultralytics opencv-python pandas

In [ ]:
import time
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from ultralytics import RTDETR

TILE_SIZE = 320
TILE_STRIDE = 260   # Solapamiento de 60px entre tiles
WINDOW_SIZE = 64

# Buscar carpeta models/ en el directorio actual o en la raíz del proyecto
models_dir = next((p for p in [Path("models"), Path("../models")] if p.exists()), None)
assert models_dir is not None, "No se encontró la carpeta 'models'."

rt_weights = models_dir / "rtdetr_fold0_best.pt"
fot_weights = models_dir / "fot_v2_fold0.pt"
assert rt_weights.exists() and fot_weights.exists(), f"Faltan los archivos .pt dentro de {models_dir.resolve()}"

use_cuda = torch.cuda.is_available()
device = torch.device("cuda:0" if use_cuda else "cpu")
yolo_device = 0 if use_cuda else "cpu"


def build_window_features(x_raw, y_raw, score_raw, valid_mask):
    W = len(x_raw)
    idx = np.arange(W)
    obs_idx = idx[valid_mask]

    if len(obs_idx) == 0:
        x_interp = np.full(W, 0.5, dtype=np.float32)
        y_interp = np.full(W, 0.5, dtype=np.float32)
        dt_prev = np.ones(W, dtype=np.float32)
        dt_next = np.ones(W, dtype=np.float32)
        has_both_anchors = np.zeros(W, dtype=np.float32)
    else:
        x_interp = np.interp(idx, obs_idx, x_raw[obs_idx]).astype(np.float32)
        y_interp = np.interp(idx, obs_idx, y_raw[obs_idx]).astype(np.float32)

        last_seen = np.full(W, -W, dtype=np.float32)
        next_seen = np.full(W, 2 * W, dtype=np.float32)
        cur = -W
        for t in range(W):
            if valid_mask[t]:
                cur = t
            last_seen[t] = cur
        cur = 2 * W
        for t in range(W - 1, -1, -1):
            if valid_mask[t]:
                cur = t
            next_seen[t] = cur

        dt_prev = np.clip((idx - last_seen) / float(W), 0.0, 1.0).astype(np.float32)
        dt_next = np.clip((next_seen - idx) / float(W), 0.0, 1.0).astype(np.float32)
        has_both_anchors = ((last_seen >= 0) & (next_seen < W)).astype(np.float32)

    vx = np.gradient(x_interp) * 10.0
    vy = np.gradient(y_interp) * 10.0
    ax = np.gradient(vx)
    ay = np.gradient(vy)
    is_obs = valid_mask.astype(np.float32)
    sc = np.where(valid_mask, score_raw, 0.0).astype(np.float32)
    gap_len = np.where(valid_mask, 0.0, np.clip(dt_prev + dt_next, 0.0, 1.0)).astype(np.float32)

    feats = np.stack([
        x_interp, y_interp, is_obs, dt_prev, dt_next,
        vx, vy, ax, ay, sc, gap_len, has_both_anchors
    ], axis=-1)
    baseline_xy = np.stack([x_interp, y_interp], axis=-1)
    return feats, baseline_xy


class FootballTransformerV2(nn.Module):
    def __init__(self, in_features=12, d_model=160, nhead=8, num_layers=4, max_len=128, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_features, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
            nn.LayerNorm(d_model),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.coord_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 2)
        )
        self.hit5_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, feats, baseline_xy):
        W = feats.shape[1]
        h = self.input_proj(feats) + self.pos_embed[:, :W, :]
        h = self.encoder(h)
        pred_xy = baseline_xy + self.coord_head(h)
        hit5_logits = self.hit5_head(h).squeeze(-1)
        return pred_xy, hit5_logits


def run_fot_v2_sequence(fot_model, raw_dets, frame_w, frame_h, device, profile="max_coverage"):
    if profile == "high_precision":
        det_input_thr, max_gap, conf_thr = 0.40, 2, 0.72
    else:
        det_input_thr, max_gap, conf_thr = 0.30, 4, 0.45

    N = len(raw_dets)
    obs_x = np.zeros(N, dtype=np.float32)
    obs_y = np.zeros(N, dtype=np.float32)
    scores = np.zeros(N, dtype=np.float32)
    valid = np.zeros(N, dtype=bool)

    for i, (_, gx, gy, sc) in enumerate(raw_dets):
        if sc >= det_input_thr and not np.isnan(gx):
            obs_x[i], obs_y[i], scores[i], valid[i] = gx / frame_w, gy / frame_h, sc, True

    max_jump_px = 160.0 * (frame_w / 1920.0)
    v_idx = np.where(valid)[0]
    for k in range(1, len(v_idx) - 1):
        i_p, i_c, i_n = v_idx[k - 1], v_idx[k], v_idx[k + 1]
        s1, s2 = max(1, i_c - i_p), max(1, i_n - i_c)
        v1 = np.hypot((obs_x[i_c] - obs_x[i_p]) * frame_w, (obs_y[i_c] - obs_y[i_p]) * frame_h) / s1
        v2 = np.hypot((obs_x[i_n] - obs_x[i_c]) * frame_w, (obs_y[i_n] - obs_y[i_c]) * frame_h) / s2
        if v1 > max_jump_px and v2 > max_jump_px and scores[i_c] < 0.72:
            valid[i_c] = False

    win_weights = np.hanning(WINDOW_SIZE + 2)[1:-1].astype(np.float32)
    acc_xy = np.zeros((N, 2), dtype=np.float32)
    acc_p5 = np.zeros(N, dtype=np.float32)
    acc_w = np.zeros(N, dtype=np.float32)

    starts = list(range(0, max(1, N - WINDOW_SIZE + 1), 16))
    if N > WINDOW_SIZE and starts[-1] != N - WINDOW_SIZE:
        starts.append(N - WINDOW_SIZE)

    with torch.no_grad():
        for s0 in starts:
            s1 = min(N, s0 + WINDOW_SIZE)
            cur_len = s1 - s0
            if cur_len < WINDOW_SIZE:
                pad = WINDOW_SIZE - cur_len
                fx = np.pad(obs_x[s0:s1], (0, pad), mode="edge")
                fy = np.pad(obs_y[s0:s1], (0, pad), mode="edge")
                fsc = np.pad(scores[s0:s1], (0, pad), mode="constant")
                fv = np.pad(valid[s0:s1], (0, pad), mode="constant")
            else:
                fx, fy, fsc, fv = obs_x[s0:s1], obs_y[s0:s1], scores[s0:s1], valid[s0:s1]

            feats, base_xy = build_window_features(fx, fy, fsc, fv)
            p_xy, h5_log = fot_model(
                torch.from_numpy(feats).unsqueeze(0).to(device),
                torch.from_numpy(base_xy).unsqueeze(0).to(device)
            )
            p_xy = p_xy.squeeze(0).cpu().numpy()[:cur_len]
            p_h5 = torch.sigmoid(h5_log).squeeze(0).cpu().numpy()[:cur_len]

            w_sl = win_weights[:cur_len]
            acc_xy[s0:s1] += p_xy * w_sl[:, None]
            acc_p5[s0:s1] += p_h5 * w_sl
            acc_w[s0:s1] += w_sl

    final_xy = acc_xy / np.maximum(acc_w[:, None], 1e-6)
    final_p5 = acc_p5 / np.maximum(acc_w, 1e-6)

    det_idx = np.where(valid)[0]
    results = {}
    for i, (fr_idx, _, _, _) in enumerate(raw_dets):
        if valid[i]:
            px, py = float(obs_x[i] * frame_w), float(obs_y[i] * frame_h)
            conf = float(max(scores[i], 0.5 * scores[i] + 0.5 * final_p5[i]))
            results[fr_idx] = (px, py, conf, "DETECTED", 0)
        else:
            pos = np.searchsorted(det_idx, i)
            if 0 < pos < len(det_idx):
                gap_steps = int(det_idx[pos] - det_idx[pos - 1] - 1)
                if gap_steps <= max_gap and final_p5[i] >= conf_thr:
                    px, py = float(final_xy[i, 0] * frame_w), float(final_xy[i, 1] * frame_h)
                    results[fr_idx] = (px, py, float(final_p5[i]), "FOT_BRIDGED", gap_steps)
                else:
                    results[fr_idx] = (np.nan, np.nan, 0.0, "NONE", gap_steps)
            else:
                results[fr_idx] = (np.nan, np.nan, 0.0, "NONE", 999)
    return results


def compute_tile_coords(frame_w, frame_h, tile_size=320, stride=260):
    xs = list(range(0, max(1, frame_w - tile_size + 1), stride))
    if xs[-1] != frame_w - tile_size:
        xs.append(max(0, frame_w - tile_size))
    ys = list(range(0, max(1, frame_h - tile_size + 1), stride))
    if ys[-1] != frame_h - tile_size:
        ys.append(max(0, frame_h - tile_size))
    return [(x, y) for y in ys for x in xs]


rt_model = RTDETR(str(rt_weights))
fot_model = FootballTransformerV2(in_features=12, d_model=160, nhead=8, num_layers=4, max_len=WINDOW_SIZE).to(device)
fot_model.load_state_dict(torch.load(fot_weights, map_location=device))
fot_model.eval()

print(f"Modelos cargados correctamente en {device} desde '{models_dir.resolve()}'")

In [ ]:
VIDEO_PATH = r"D:\Proyectos\Barcelona project local\data\tactical\BAR-ATH_tactical.mp4"
HOMOGRAPHY_PATH = r"D:\Proyectos\Barcelona project local\data\tactical\BAR-ATH_tactical_homography.csv"

In [ ]:
import random

# ==============================================================================
# 1. CONFIGURACIÓN REPRODUCIBLE Y PARÁMETROS DE MEMORIA / ANTI-BOTINES
# ==============================================================================


RANDOM_SEED = 0             # Reproducibilidad exacta de la jugada
HALF_CHOICE = "1st_half"     # "1st_half", "2nd_half", o "any_half"
CLIP_SECONDS = 30.0
PROFILE = "max_coverage"

MAX_REPROJ_ERROR = 5.0
MARGIN = 0.05                # +5% de margen en largo y ancho -> [-0.05, 1.05]

# Parámetros de tracking y memoria a 25 fps
DET_CONF_FLOOR = 0.07        # Umbral bajo para rescatar balones parcialmente tapados cerca del track
FAR_CONF_FLOOR = 0.18        # Umbral más estricto fuera de la zona del balón (filtra botines lejanos)
MAX_BALL_SPEED_MPS = 36.0    # Velocidad física máxima (~130 km/h)
BASE_GATE_RADIUS_M = 3.2     # Radio base de regate/rebote en metros
JUMP_CONFIRM_FRAMES = 3      # Frames consecutivos requeridos para aceptar un salto fuera del radio cinemático
MAX_KINEMATIC_GAP_S = 1.4    # Memoria máxima (segundos) para reconstruir oclusiones largas en espacio de cancha

MATCH_ROIS = {
    "before_match": (0, 11184),
    "1st_half":     (11185, 90651),
    "half_time":    (90652, 110807),
    "2nd_half":     (110808, 190197),
    "after":        (190198, 199318),
}

MATRIX_COLUMNS = ["h{}{}".format(row, col) for row in range(3) for col in range(3)]


def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_global_seed(RANDOM_SEED)


# ==============================================================================
# 2. FUNCIONES DE HOMOGRAFÍA Y MÁSCARAS
# ==============================================================================
def load_homography(path: str):
    table = pd.read_csv(path)
    missing = [name for name in ["frame", "reprojection_error", "valid"] + MATRIX_COLUMNS
               if name not in table.columns]
    if missing:
        raise ValueError(f"{path}: missing columns {missing}")
    matrices = table[MATRIX_COLUMNS].values.astype(np.float64).reshape(-1, 3, 3)
    return matrices, table["reprojection_error"].values, table["valid"].values.astype(bool)


def project(matrix, x, y):
    x, y = np.asarray(x, dtype=np.float64), np.asarray(y, dtype=np.float64)
    denominator = matrix[2, 0] * x + matrix[2, 1] * y + matrix[2, 2]
    return ((matrix[0, 0] * x + matrix[0, 1] * y + matrix[0, 2]) / denominator,
            (matrix[1, 0] * x + matrix[1, 1] * y + matrix[1, 2]) / denominator)


def unproject(matrix, pixel_x, pixel_y):
    return project(np.linalg.inv(matrix), pixel_x, pixel_y)


def get_valid_H(fr_idx: int, matrices: np.ndarray, reproj_err: np.ndarray, valid: np.ndarray):
    if 0 <= fr_idx < len(valid) and valid[fr_idx] and reproj_err[fr_idx] <= MAX_REPROJ_ERROR:
        H = matrices[fr_idx]
        if not np.isnan(H).any():
            return H
    return None


def densify_rect_norm(x0: float, y0: float, x1: float, y1: float, steps: int = 32):
    xs = np.concatenate([
        np.linspace(x0, x1, steps, endpoint=False), np.full(steps, x1),
        np.linspace(x1, x0, steps, endpoint=False), np.full(steps, x0),
    ])
    ys = np.concatenate([
        np.full(steps, y0), np.linspace(y0, y1, steps, endpoint=False),
        np.full(steps, y1), np.linspace(y1, y0, steps, endpoint=False),
    ])
    return xs, ys


def project_polygon_px(H: np.ndarray, xs: np.ndarray, ys: np.ndarray):
    denom = H[2, 0] * xs + H[2, 1] * ys + H[2, 2]
    center_denom = H[2, 0] * 0.5 + H[2, 1] * 0.5 + H[2, 2]
    same_side = (denom * center_denom) > 1e-8
    if same_side.sum() < 3:
        return None
    px, py = project(H, xs[same_side], ys[same_side])
    return np.round(np.column_stack([px, py])).astype(np.int32)


def build_pitch_masks(H: np.ndarray, frame_w: int, frame_h: int, margin: float = 0.05):
    mask_margin = np.zeros((frame_h, frame_w), dtype=np.uint8)
    mask_exact = np.zeros((frame_h, frame_w), dtype=np.uint8)

    xm, ym = densify_rect_norm(-margin, -margin, 1.0 + margin, 1.0 + margin, steps=32)
    poly_margin = project_polygon_px(H, xm, ym)
    if poly_margin is not None:
        cv2.fillPoly(mask_margin, [poly_margin], 255)

    xe, ye = densify_rect_norm(0.0, 0.0, 1.0, 1.0, steps=32)
    poly_exact = project_polygon_px(H, xe, ye)
    if poly_exact is not None:
        cv2.fillPoly(mask_exact, [poly_exact], 255)

    return mask_margin, mask_exact


# ==============================================================================
# 3. NMS DE CANDIDATOS + GATING CINEMÁTICO ANTI-BOTINES
# ==============================================================================
def nms_candidates(cands: list, min_dist_px: float = 14.0) -> list:
    """Agrupa detecciones duplicadas de tiles solapados conservando la de mayor confianza."""
    if not cands:
        return []
    cands_sorted = sorted(cands, key=lambda c: c[2], reverse=True)
    kept = []
    for gx, gy, sc in cands_sorted:
        if all(np.hypot(gx - kx, gy - ky) >= min_dist_px for kx, ky, _ in kept):
            kept.append((gx, gy, sc))
    return kept


class PitchKinematicGate:
    """
    Filtra falsos positivos de botines usando cinemática en metros sobre la cancha.
    - Si el candidato cae dentro del radio físico alcanzable desde el último balón visto,
      combina confianza visual + cercanía espacial.
    - Si aparece un candidato lejano (ej. un botín en otra zona), exige JUMP_CONFIRM_FRAMES
      frames coherentes antes de cambiar el track.
    """
    def __init__(self, fps: float):
        self.fps = fps
        self.dt = 1.0 / fps
        self.last_fr = None
        self.last_mx = None
        self.last_my = None
        self.vx = 0.0  # m/s
        self.vy = 0.0  # m/s
        self.jump_buffer = []  # [(fr, gx, gy, sc, mx, my)]

    def _to_meters(self, H, gx, gy):
        if H is None:
            return None, None
        nx, ny = unproject(H, gx, gy)
        return float(nx) * 105.0, float(ny) * 68.0

    def _update_track(self, fr: int, mx: float, my: float):
        if self.last_fr is not None and mx is not None:
            dt_sec = max(1, fr - self.last_fr) * self.dt
            if dt_sec <= 0.4:
                new_vx = np.clip((mx - self.last_mx) / dt_sec, -MAX_BALL_SPEED_MPS, MAX_BALL_SPEED_MPS)
                new_vy = np.clip((my - self.last_my) / dt_sec, -MAX_BALL_SPEED_MPS, MAX_BALL_SPEED_MPS)
                self.vx = 0.65 * self.vx + 0.35 * new_vx
                self.vy = 0.65 * self.vy + 0.35 * new_vy
            else:
                self.vx, self.vy = 0.0, 0.0
        self.last_fr = fr
        self.last_mx = mx
        self.last_my = my
        self.jump_buffer.clear()

    def step(self, fr: int, candidates: list, H: np.ndarray, det_history: dict):
        if not candidates or H is None:
            if not candidates:
                return np.nan, np.nan, 0.0
            gx, gy, sc = candidates[0]
            return (gx, gy, sc) if sc >= FAR_CONF_FLOOR else (np.nan, np.nan, 0.0)

        enriched = []
        for gx, gy, sc in candidates:
            mx, my = self._to_meters(H, gx, gy)
            enriched.append((gx, gy, sc, mx, my))

        # Si no hay track activo reciente (> 1.5s sin ver el balón), exigir confianza mínima o confirmación
        has_active_track = (
            self.last_fr is not None
            and self.last_mx is not None
            and (fr - self.last_fr) <= int(round(1.5 * self.fps))
        )

        if has_active_track:
            gap_frames = max(1, fr - self.last_fr)
            dt_sec = gap_frames * self.dt
            damp = 0.90 ** gap_frames
            pred_mx = self.last_mx + self.vx * dt_sec * damp
            pred_my = self.last_my + self.vy * dt_sec * damp
            max_reach_m = BASE_GATE_RADIUS_M + MAX_BALL_SPEED_MPS * dt_sec
            sigma_m = 1.6 + 0.45 * gap_frames

            in_gate = []
            for gx, gy, sc, mx, my in enriched:
                d_m = np.hypot(mx - pred_mx, my - pred_my)
                if d_m <= max_reach_m:
                    spatial_w = float(np.exp(-0.5 * (d_m / sigma_m) ** 2))
                    eff_score = sc * (0.35 + 0.65 * spatial_w)
                    in_gate.append((eff_score, sc, gx, gy, mx, my))

            if in_gate:
                in_gate.sort(key=lambda item: item[0], reverse=True)
                _, best_sc, best_gx, best_gy, best_mx, best_my = in_gate[0]
                self._update_track(fr, best_mx, best_my)
                return best_gx, best_gy, best_sc

        # Candidatos fuera del radio cinemático (posible botín lejano o cambio real de balón)
        far_cands = [c for c in enriched if c[2] >= FAR_CONF_FLOOR]
        if not far_cands:
            return np.nan, np.nan, 0.0

        # Si hay una detección inequívoca (ej. balón aéreo alto > 0.68), aceptar de inmediato
        top_far = far_cands[0]
        if top_far[2] >= 0.68:
            self._update_track(fr, top_far[3], top_far[4])
            return top_far[0], top_far[1], top_far[2]

        # Confirmación temporal de JUMP_CONFIRM_FRAMES para descartar botines intermitentes
        self.jump_buffer = [b for b in self.jump_buffer if (fr - b[0]) <= JUMP_CONFIRM_FRAMES + 1]
        best_cluster = None
        for gx, gy, sc, mx, my in far_cands:
            matches = [b for b in self.jump_buffer if np.hypot(mx - b[4], my - b[5]) <= 3.5]
            if best_cluster is None or len(matches) > len(best_cluster[1]):
                best_cluster = ((fr, gx, gy, sc, mx, my), matches)

        cur_item, matches = best_cluster
        self.jump_buffer.append(cur_item)

        if len(matches) + 1 >= JUMP_CONFIRM_FRAMES:
            # Confirmado: rescatar retroactivamente los frames del buffer en det_history
            for b_fr, b_gx, b_gy, b_sc, _, _ in matches:
                det_history[b_fr] = (b_fr, b_gx, b_gy, b_sc)
            self._update_track(fr, cur_item[4], cur_item[5])
            return cur_item[1], cur_item[2], cur_item[3]

        return np.nan, np.nan, 0.0


# ==============================================================================
# 4. INFERENCIA A 25 FPS COMPLETOS + DOBLE FASE FoT v2 + MEMORIA EN CANCHA
# ==============================================================================
assert Path(VIDEO_PATH).exists(), f"Video not found: {VIDEO_PATH}"
assert Path(HOMOGRAPHY_PATH).exists(), f"Homography not found: {HOMOGRAPHY_PATH}"

matrices, reproj_err, valid = load_homography(HOMOGRAPHY_PATH)

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
vid_total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

rng = random.Random(RANDOM_SEED)
chosen_half = rng.choice(["1st_half", "2nd_half"]) if HALF_CHOICE == "any_half" else HALF_CHOICE
roi_start, roi_end = MATCH_ROIS[chosen_half]
roi_end = min(roi_end, vid_total_frames - 1)

clip_len_frames = int(round(CLIP_SECONDS * fps))
max_start = max(roi_start, roi_end - clip_len_frames)

best_start = rng.randint(roi_start, max_start)
for _ in range(40):
    cand_start = rng.randint(roi_start, max_start)
    sl = slice(cand_start, min(len(valid), cand_start + clip_len_frames))
    if valid[sl].mean() >= 0.75:
        best_start = cand_start
        break

start_frame = best_start
end_frame = min(roi_end, start_frame + clip_len_frames - 1)
print(f"Seed={RANDOM_SEED} | [{chosen_half}] frames [{start_frame} .. {end_frame}] @ {fps:.1f} fps (full-rate)")

cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
all_tiles = compute_tile_coords(frame_w, frame_h, TILE_SIZE, TILE_STRIDE)
gate = PitchKinematicGate(fps=fps)

det_history = {}
t0 = time.perf_counter()

# Procesar TODOS los frames a 25 fps (step = 1)
for fr_idx in range(start_frame, end_frame + 1):
    ret, frame = cap.read()
    if not ret:
        break

    H_cur = get_valid_H(fr_idx, matrices, reproj_err, valid)
    if H_cur is not None:
        mask_margin, _ = build_pitch_masks(H_cur, frame_w, frame_h, margin=MARGIN)
        masked_frame = cv2.bitwise_and(frame, frame, mask=mask_margin)
        active_tiles = [
            (tx, ty) for (tx, ty) in all_tiles
            if mask_margin[ty:ty + TILE_SIZE, tx:tx + TILE_SIZE].any()
        ]
    else:
        mask_margin = None
        masked_frame = frame
        active_tiles = all_tiles

    if len(active_tiles) == 0:
        det_history[fr_idx] = (fr_idx, np.nan, np.nan, 0.0)
        continue

    crops = [masked_frame[ty:ty + TILE_SIZE, tx:tx + TILE_SIZE] for (tx, ty) in active_tiles]
    results = rt_model.predict(
        source=crops, imgsz=TILE_SIZE, conf=DET_CONF_FLOOR, max_det=4, device=yolo_device, verbose=False
    )

    frame_cands = []
    for (tx, ty), r in zip(active_tiles, results):
        if r.boxes is not None and len(r.boxes) > 0:
            confs = r.boxes.conf.cpu().numpy()
            xywhn = r.boxes.xywhn.cpu().numpy()
            for i in range(len(confs)):
                sc = float(confs[i])
                gx = tx + float(xywhn[i, 0]) * TILE_SIZE
                gy = ty + float(xywhn[i, 1]) * TILE_SIZE
                if mask_margin is not None:
                    ix = min(max(int(round(gx)), 0), frame_w - 1)
                    iy = min(max(int(round(gy)), 0), frame_h - 1)
                    if mask_margin[iy, ix] == 0:
                        continue
                frame_cands.append((gx, gy, sc))

    frame_cands = nms_candidates(frame_cands, min_dist_px=14.0)
    gx, gy, sc = gate.step(fr_idx, frame_cands, H_cur, det_history)
    det_history[fr_idx] = (fr_idx, gx, gy, sc)

    if (fr_idx - start_frame + 1) % 125 == 0:
        print(f"  Procesados {fr_idx - start_frame + 1}/{end_frame - start_frame + 1} frames | {time.perf_counter() - t0:.1f}s")

cap.release()

# ==============================================================================
# 5. FUSIÓN RTS EN CANCHA + PODA DE PICOS + RECONSTRUCCIÓN ACOTADA
# ==============================================================================
MAX_LATERAL_DEV_M = 0.65     # Desviación curva máxima permitida respecto a línea recta (metros)
SPIKE_SPEED_MPS = 32.0       # Umbral para detectar anclas aisladas de 1-2 frames (botines)

ordered_dets = [det_history[f] for f in sorted(det_history.keys())]
fot_even = run_fot_v2_sequence(fot_model, ordered_dets[0::2], frame_w, frame_h, device, profile=PROFILE)
fot_odd = run_fot_v2_sequence(fot_model, ordered_dets[1::2], frame_w, frame_h, device, profile=PROFILE)

raw_merged = {}
raw_merged.update(fot_even)
raw_merged.update(fot_odd)
all_f = sorted(raw_merged.keys())

# --- Paso A: Convertir a metros sobre cancha y podar anclas aisladas imposibles ---
pitch_pts = {}  # fr -> [mx, my, conf, state, gap]
for f in all_f:
    px, py, conf, st, gap = raw_merged[f]
    H_f = get_valid_H(f, matrices, reproj_err, valid)
    if st != "NONE" and not np.isnan(px) and H_f is not None:
        nx, ny = unproject(H_f, px, py)
        pitch_pts[f] = [float(nx) * 105.0, float(ny) * 68.0, float(conf), st, int(gap)]
    else:
        pitch_pts[f] = None

valid_f = [f for f in all_f if pitch_pts[f] is not None]
for i in range(1, len(valid_f) - 1):
    f_prev, f_cur, f_next = valid_f[i - 1], valid_f[i], valid_f[i + 1]
    dt1 = (f_cur - f_prev) / fps
    dt2 = (f_next - f_cur) / fps
    dt_tot = (f_next - f_prev) / fps
    if dt1 <= 0.25 and dt2 <= 0.25:
        p0, p1, p2 = pitch_pts[f_prev], pitch_pts[f_cur], pitch_pts[f_next]
        v_in = np.hypot(p1[0] - p0[0], p1[1] - p0[1]) / max(dt1, 1e-3)
        v_out = np.hypot(p2[0] - p1[0], p2[1] - p1[1]) / max(dt2, 1e-3)
        v_direct = np.hypot(p2[0] - p0[0], p2[1] - p0[1]) / max(dt_tot, 1e-3)
        # Si entra y sale a velocidad irreal pero los extremos son coherentes, es un falso positivo aislado
        if v_in > SPIKE_SPEED_MPS and v_out > SPIKE_SPEED_MPS and v_direct < 22.0:
            pitch_pts[f_cur] = None

# --- Paso B: Suavizador Kalman Bidireccional (RTS) en metros para eliminar zig-zag par/impar ---
def rts_smooth_segment(seg_frames: list, pts_dict: dict, fps: float):
    """Suaviza (mx, my) con modelo de velocidad constante y ruido diferenciado por estado."""
    n = len(seg_frames)
    if n < 3:
        return
    dt = 1.0 / fps
    F = np.array([[1, 0, dt, 0],
                  [0, 1, 0, dt],
                  [0, 0, 1,  0],
                  [0, 0, 0,  1]], dtype=np.float64)
    H_obs = np.array([[1, 0, 0, 0],
                      [0, 1, 0, 0]], dtype=np.float64)
    # Aceleración de proceso (permite cambios reales de ritmo en pases sin zig-zag)
    q = 18.0
    G = np.array([[0.5 * dt**2, 0],
                  [0, 0.5 * dt**2],
                  [dt, 0],
                  [0, dt]], dtype=np.float64)
    Q = (G @ G.T) * q

    xs_f, Ps_f, xs_p, Ps_p = [], [], [], []
    p0 = pts_dict[seg_frames[0]]
    x = np.array([p0[0], p0[1], 0.0, 0.0], dtype=np.float64)
    P = np.diag([0.5, 0.5, 25.0, 25.0])

    for k, fr in enumerate(seg_frames):
        if k > 0:
            step_k = max(1, fr - seg_frames[k - 1])
            Fk = np.eye(4)
            Fk[0, 2] = step_k * dt
            Fk[1, 3] = step_k * dt
            x = Fk @ x
            P = Fk @ P @ Fk.T + Q * step_k
        xs_p.append(x.copy())
        Ps_p.append(P.copy())

        obs = pts_dict[fr]
        # Mayor confianza en detecciones directas RT-DETR; más flexible en FoT_BRIDGED
        r_std = 0.22 if obs[3] == "DETECTED" else 0.85
        R = np.eye(2) * (r_std ** 2)
        z = np.array([obs[0], obs[1]], dtype=np.float64)
        y_res = z - H_obs @ x
        S = H_obs @ P @ H_obs.T + R
        K = P @ H_obs.T @ np.linalg.inv(S)
        x = x + K @ y_res
        P = (np.eye(4) - K @ H_obs) @ P
        xs_f.append(x.copy())
        Ps_f.append(P.copy())

    # Pasada hacia atrás (RTS)
    xs_s = [None] * n
    xs_s[-1] = xs_f[-1]
    for k in range(n - 2, -1, -1):
        step_k = max(1, seg_frames[k + 1] - seg_frames[k])
        Fk = np.eye(4)
        Fk[0, 2] = step_k * dt
        Fk[1, 3] = step_k * dt
        C = Ps_f[k] @ Fk.T @ np.linalg.inv(Ps_p[k + 1])
        xs_s[k] = xs_f[k] + C @ (xs_s[k + 1] - xs_p[k + 1])

    for k, fr in enumerate(seg_frames):
        pts_dict[fr][0] = float(xs_s[k][0])
        pts_dict[fr][1] = float(xs_s[k][1])

# Aplicar RTS sobre cada bloque continuo de tracking activo
cur_seg = []
for f in all_f:
    if pitch_pts[f] is not None:
        cur_seg.append(f)
    else:
        if len(cur_seg) >= 3:
            rts_smooth_segment(cur_seg, pitch_pts, fps)
        cur_seg = []
if len(cur_seg) >= 3:
    rts_smooth_segment(cur_seg, pitch_pts, fps)

# --- Paso C: Reconstrucción de huecos con límite estricto de curvatura lateral ---
max_gap_frames = int(round(MAX_KINEMATIC_GAP_S * fps))
idx_ptr = 0

while idx_ptr < len(all_f):
    if pitch_pts[all_f[idx_ptr]] is not None:
        idx_ptr += 1
        continue

    gap_start = idx_ptr
    while idx_ptr < len(all_f) and pitch_pts[all_f[idx_ptr]] is None:
        idx_ptr += 1
    gap_end = idx_ptr - 1

    if gap_start > 0 and idx_ptr < len(all_f):
        f_prev = all_f[gap_start - 1]
        f_next = all_f[idx_ptr]
        gap_len = f_next - f_prev - 1
        dt_span_s = (f_next - f_prev) / fps

        if 1 <= gap_len <= max_gap_frames:
            p_prev = pitch_pts[f_prev]
            p_next = pitch_pts[f_next]
            dx_chord = p_next[0] - p_prev[0]
            dy_chord = p_next[1] - p_prev[1]
            chord_m = np.hypot(dx_chord, dy_chord)

            # Verificar velocidad media físicamente plausible (<= 35 m/s)
            if (chord_m / max(dt_span_s, 1e-3)) <= 35.0:
                # Estimar velocidades de entrada y salida usando ventana robusta de hasta 4 frames
                w_in = [all_f[j] for j in range(max(0, gap_start - 5), gap_start) if pitch_pts[all_f[j]] is not None]
                w_out = [all_f[j] for j in range(idx_ptr, min(len(all_f), idx_ptr + 5)) if pitch_pts[all_f[j]] is not None]

                if len(w_in) >= 2:
                    dt_in = (w_in[-1] - w_in[0]) / fps
                    vx0 = (pitch_pts[w_in[-1]][0] - pitch_pts[w_in[0]][0]) / max(dt_in, 1e-3)
                    vy0 = (pitch_pts[w_in[-1]][1] - pitch_pts[w_in[0]][1]) / max(dt_in, 1e-3)
                else:
                    vx0, vy0 = dx_chord / dt_span_s, dy_chord / dt_span_s

                if len(w_out) >= 2:
                    dt_out = (w_out[-1] - w_out[0]) / fps
                    vx1 = (pitch_pts[w_out[-1]][0] - pitch_pts[w_out[0]][0]) / max(dt_out, 1e-3)
                    vy1 = (pitch_pts[w_out[-1]][1] - pitch_pts[w_out[0]][1]) / max(dt_out, 1e-3)
                else:
                    vx1, vy1 = dx_chord / dt_span_s, dy_chord / dt_span_s

                # Tangentes amortiguadas para evitar bucles en pases/rebotes
                m0x, m0y = vx0 * dt_span_s * 0.65, vy0 * dt_span_s * 0.65
                m1x, m1y = vx1 * dt_span_s * 0.65, vy1 * dt_span_s * 0.65

                ts = np.array([(all_f[k] - f_prev) / float(f_next - f_prev) for k in range(gap_start, gap_end + 1)])
                h00 = 2 * ts**3 - 3 * ts**2 + 1
                h10 = ts**3 - 2 * ts**2 + ts
                h01 = -2 * ts**3 + 3 * ts**2
                h11 = ts**3 - ts**2

                hx = h00 * p_prev[0] + h10 * m0x + h01 * p_next[0] + h11 * m1x
                hy = h00 * p_prev[1] + h10 * m0y + h01 * p_next[1] + h11 * m1y

                # Línea recta de referencia (cuerda en el plano de la cancha)
                lx = p_prev[0] + ts * dx_chord
                ly = p_prev[1] + ts * dy_chord

                # Medir desviación lateral máxima y escalar si supera MAX_LATERAL_DEV_M
                dev_m = np.hypot(hx - lx, hy - ly)
                max_dev = float(dev_m.max()) if len(dev_m) > 0 else 0.0
                blend_curve = min(1.0, MAX_LATERAL_DEV_M / max(max_dev, 1e-6))

                final_mx = lx + blend_curve * (hx - lx)
                final_my = ly + blend_curve * (hy - ly)
                c_base = min(p_prev[2], p_next[2]) * 0.85

                for idx_local, k in enumerate(range(gap_start, gap_end + 1)):
                    fk = all_f[k]
                    pitch_pts[fk] = [float(final_mx[idx_local]), float(final_my[idx_local]), c_base, "KINEMATIC_BRIDGED", gap_len]

# --- Paso D: Reproyectar todas las coordenadas suaves de cancha a píxeles de video ---
full_frame_preds = {}
for f in all_f:
    pt = pitch_pts[f]
    H_f = get_valid_H(f, matrices, reproj_err, valid)
    if pt is not None and H_f is not None:
        px_f, py_f = project(H_f, pt[0] / 105.0, pt[1] / 68.0)
        full_frame_preds[f] = (float(px_f), float(py_f), pt[2], pt[3], pt[4])
    else:
        full_frame_preds[f] = (np.nan, np.nan, 0.0, "NONE", 999)

n_det = sum(1 for v in full_frame_preds.values() if v[3] == "DETECTED")
n_fot = sum(1 for v in full_frame_preds.values() if v[3] == "FOT_BRIDGED")
n_kin = sum(1 for v in full_frame_preds.values() if v[3] == "KINEMATIC_BRIDGED")
n_none = sum(1 for v in full_frame_preds.values() if v[3] == "NONE")
cov_pct = 100.0 * (len(full_frame_preds) - n_none) / max(1, len(full_frame_preds))

print(f"\nListo en {time.perf_counter() - t0:.1f}s | Cobertura total: {cov_pct:.1f}% ({len(full_frame_preds) - n_none}/{len(full_frame_preds)} frames)")
print(f"  RT-DETR filtrado : {n_det}")
print(f"  FoT v2 (25fps)   : +{n_fot}")
print(f"  Memoria Cancha   : +{n_kin}")
print(f"  Sin track (NONE) : {n_none}")

In [ ]:
OUTPUT_VIDEO_PATH = f"play_{chosen_half}_seed{RANDOM_SEED}_{start_frame}_{end_frame}.mp4"
OUTPUT_CSV_PATH = f"play_{chosen_half}_seed{RANDOM_SEED}_{start_frame}_{end_frame}.csv"

RENDER_BLACK_MASK = True
PITCH_PLANE_COLOR = (220, 50, 220)
PITCH_LINE_COLOR = (255, 240, 255)
PITCH_PLANE_ALPHA = 0.24


def draw_pitch_overlay(frame: np.ndarray, H: np.ndarray,
                       mask_margin: np.ndarray, mask_exact: np.ndarray,
                       black_outside: bool = True,
                       plane_color=(220, 50, 220),
                       line_color=(255, 240, 255),
                       alpha: float = 0.24) -> np.ndarray:
    out = cv2.bitwise_and(frame, frame, mask=mask_margin) if black_outside else frame.copy()

    idx_plane = mask_exact > 0
    if np.any(idx_plane):
        tint = np.full_like(out[idx_plane], plane_color, dtype=np.uint8)
        out[idx_plane] = cv2.addWeighted(out[idx_plane], 1.0 - alpha, tint, alpha, 0)

    def seg(x0, y0, x1, y1, steps=24):
        return np.linspace(x0, x1, steps), np.linspace(y0, y1, steps)

    pa_x = 16.5 / 105.0
    pa_y0, pa_y1 = 0.5 - 20.16 / 68.0, 0.5 + 20.16 / 68.0
    ga_x = 5.5 / 105.0
    ga_y0, ga_y1 = 0.5 - 9.16 / 68.0, 0.5 + 9.16 / 68.0
    theta = np.linspace(0, 2 * np.pi, 72)

    lines_norm = [
        seg(0.0, 0.0, 1.0, 0.0), seg(1.0, 0.0, 1.0, 1.0),
        seg(1.0, 1.0, 0.0, 1.0), seg(0.0, 1.0, 0.0, 0.0),
        seg(0.5, 0.0, 0.5, 1.0),
        (0.5 + (9.15 / 105.0) * np.cos(theta), 0.5 + (9.15 / 68.0) * np.sin(theta)),
        seg(0.0, pa_y0, pa_x, pa_y0), seg(pa_x, pa_y0, pa_x, pa_y1), seg(pa_x, pa_y1, 0.0, pa_y1),
        seg(1.0, pa_y0, 1.0 - pa_x, pa_y0), seg(1.0 - pa_x, pa_y0, 1.0 - pa_x, pa_y1), seg(1.0 - pa_x, pa_y1, 1.0, pa_y1),
        seg(0.0, ga_y0, ga_x, ga_y0), seg(ga_x, ga_y0, ga_x, ga_y1), seg(ga_x, ga_y1, 0.0, ga_y1),
        seg(1.0, ga_y0, 1.0 - ga_x, ga_y0), seg(1.0 - ga_x, ga_y0, 1.0 - ga_x, ga_y1), seg(1.0 - ga_x, ga_y1, 1.0, ga_y1),
    ]

    center_denom = H[2, 0] * 0.5 + H[2, 1] * 0.5 + H[2, 2]
    for lx, ly in lines_norm:
        denom = H[2, 0] * lx + H[2, 1] * ly + H[2, 2]
        same_side = (denom * center_denom) > 1e-8
        if same_side.sum() >= 2:
            px_l, py_l = project(H, lx[same_side], ly[same_side])
            pts = np.round(np.column_stack([px_l, py_l])).astype(np.int32).reshape(-1, 1, 2)
            cv2.polylines(out, [pts], isClosed=False, color=line_color, thickness=2, lineType=cv2.LINE_AA)

    for sx, sy in [(0.5, 0.5), (11.0 / 105.0, 0.5), (1.0 - 11.0 / 105.0, 0.5)]:
        denom_s = H[2, 0] * sx + H[2, 1] * sy + H[2, 2]
        if (denom_s * center_denom) > 1e-8:
            cx_s, cy_s = project(H, sx, sy)
            if 0 <= cx_s < out.shape[1] and 0 <= cy_s < out.shape[0]:
                cv2.circle(out, (int(round(cx_s)), int(round(cy_s))), 4, line_color, -1, lineType=cv2.LINE_AA)

    contours, _ = cv2.findContours(mask_margin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, (160, 240, 240), 1, lineType=cv2.LINE_AA)
    return out


cap = cv2.VideoCapture(VIDEO_PATH)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, cv2.VideoWriter_fourcc(*"mp4v"), fps, (frame_w, frame_h))

STATE_COLORS = {
    "DETECTED":          (0, 230, 118),   # Verde: detección RT-DETR validada por cinemática
    "FOT_BRIDGED":       (255, 215, 0),   # Cian/dorado: reconstrucción temporal FoT v2
    "KINEMATIC_BRIDGED": (0, 140, 255),   # Naranja: memoria extendida en coordenadas de cancha
}

trail = []
csv_rows = []
prev_xy = None

for fr_idx in range(start_frame, end_frame + 1):
    ret, frame = cap.read()
    if not ret:
        break

    H_cur = get_valid_H(fr_idx, matrices, reproj_err, valid)
    if H_cur is not None:
        mask_margin, mask_exact = build_pitch_masks(H_cur, frame_w, frame_h, margin=MARGIN)
        frame = draw_pitch_overlay(
            frame, H_cur, mask_margin, mask_exact,
            black_outside=RENDER_BLACK_MASK,
            plane_color=PITCH_PLANE_COLOR,
            line_color=PITCH_LINE_COLOR,
            alpha=PITCH_PLANE_ALPHA,
        )

    px, py, conf, state, gap = full_frame_preds.get(fr_idx, (np.nan, np.nan, 0.0, "NONE", 999))
    speed_px_s = 0.0
    pitch_x_norm, pitch_y_norm = np.nan, np.nan
    pitch_x_m, pitch_y_m = np.nan, np.nan

    if state != "NONE" and not np.isnan(px):
        if prev_xy is not None:
            speed_px_s = np.hypot(px - prev_xy[0], py - prev_xy[1]) * fps
        prev_xy = (px, py)

        if H_cur is not None:
            nx, ny = unproject(H_cur, px, py)
            pitch_x_norm, pitch_y_norm = float(nx), float(ny)
            pitch_x_m, pitch_y_m = float(nx * 105.0), float(ny * 68.0)

        trail.append((int(round(px)), int(round(py)), state))
        if len(trail) > 22:
            trail.pop(0)
    else:
        prev_xy = None
        if trail:
            trail.pop(0)

    for t_i in range(1, len(trail)):
        pt1 = (trail[t_i - 1][0], trail[t_i - 1][1])
        pt2 = (trail[t_i][0], trail[t_i][1])
        if np.hypot(pt2[0] - pt1[0], pt2[1] - pt1[1]) < 150:
            col = STATE_COLORS.get(trail[t_i][2], (0, 230, 118))
            cv2.line(frame, pt1, pt2, col, max(1, int(round(3 * (t_i / len(trail))))), lineType=cv2.LINE_AA)

    if state != "NONE" and not np.isnan(px):
        cx_i, cy_i = int(round(px)), int(round(py))
        color = STATE_COLORS.get(state, (0, 230, 118))
        if state == "DETECTED":
            tag = f"BALL {conf:.2f}"
        elif state == "FOT_BRIDGED":
            tag = f"FoT {conf:.2f} (g{gap})"
        else:
            tag = f"MEM {conf:.2f} (g{gap})"

        if not np.isnan(pitch_x_norm):
            tag += f" ({pitch_x_norm:.2f}, {pitch_y_norm:.2f})"

        cv2.circle(frame, (cx_i, cy_i), 11, color, 2, lineType=cv2.LINE_AA)
        cv2.circle(frame, (cx_i, cy_i), 2, (255, 255, 255), -1)
        cv2.putText(frame, tag, (cx_i + 14, cy_i + 5), cv2.FONT_HERSHEY_SIMPLEX, 0.52, color, 2, cv2.LINE_AA)

    hud = f"[seed={RANDOM_SEED} | {chosen_half}] Fr {fr_idx} | {fr_idx/fps:06.2f}s | {state:17s} | {speed_px_s:5.0f} px/s"
    cv2.rectangle(frame, (12, 12), (820, 68), (15, 15, 15), -1)
    cv2.putText(frame, hud, (20, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (240, 240, 240), 1, cv2.LINE_AA)
    cv2.putText(
        frame,
        "Verde: RT-DETR | Dorado: FoT v2 (25fps) | Naranja: Memoria Cancha (Hermite) | Loss="
        f"{100.0 * n_none / max(1, len(full_frame_preds)):.1f}%",
        (20, 58), cv2.FONT_HERSHEY_SIMPLEX, 0.44, (220, 220, 220), 1, cv2.LINE_AA
    )

    writer.write(frame)
    csv_rows.append((
        fr_idx, round(fr_idx / fps, 4), px, py,
        pitch_x_norm, pitch_y_norm, pitch_x_m, pitch_y_m,
        conf, state, gap, speed_px_s
    ))

cap.release()
writer.release()

df_out = pd.DataFrame(
    csv_rows,
    columns=[
        "frame", "time_s", "pred_x", "pred_y",
        "pitch_x_norm", "pitch_y_norm", "pitch_x_m", "pitch_y_m",
        "score", "state", "gap_steps", "speed_px_s"
    ]
)
df_out.to_csv(OUTPUT_CSV_PATH, index=False)

print(f"Video guardado: {OUTPUT_VIDEO_PATH}")
print(f"CSV guardado  : {OUTPUT_CSV_PATH}")
print("\nDistribución de estados (25 fps completos):")
print(df_out["state"].value_counts())